ACTUALIZA LAS PROPIEDADES QUE TIENEN MIDIFICACIONES

In [1]:
import ee
ee.Authenticate()

Enter verification code:  4/1Aci98E-bqkJeF04ooXFNKwRfdZG9lqBYBunyCk9kdFq-HHH9iUajo6BK0LM



Successfully saved authorization token.


In [26]:
import ee
import geemap
# metodo de clasificacion
import jenkspy

import geopandas as gpd
from geopandas.tools import overlay
from shapely.geometry import Polygon

import pandas as pd
import numpy as np

import openpyxl

In [3]:
ee.Initialize()

In [28]:
ruta_catastro='projects/ee-bismarksr17/assets/MONTE_VERDE_CANHA_UTM'
ruta_ndvi = 'projects/ee-bismarksr17/assets/NDVI_MONTE_VERDE'

CODIGO_PROPIEDAD='chacra'
NOM_PROPIEDAD='chacra'
CODIGO_CANHERO=''
NOM_CANHERO=''

VARIEDAD='variedad'
ESTADO='RENOVACION'
SUPERFICIE='ha_parcela'

In [29]:
def asig_cat(label):
    if label == 1:
        return 0
    else:
        return label * 10

vis_params_caña = {
    'color': 'red', 
    'width': 2,
    'lineType': 'solid',
    'fillColor': '00000000',
}

# parametro de visualizacion LOTES RENOVACION
vis_params_renov = {
    'color': 'blue', 
    'width': 2,
    'lineType': 'solid',
    'fillColor': '00000000',
}

In [30]:
path_cat = r'C:\Documents\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - OTROS DATOS\PETROQUIM\SHP\MONTE_VERDE_CANHA_UTM.shp'
gdf_cat = gpd.read_file(path_cat)

In [31]:
lista_props = list(set(gdf_cat['chacra']))
print(lista_props)

['BENITEZ 2', 'EX CHAVEZ', 'SARUBI', 'BENITEZ 1', 'JARANA', 'BUENA VISTA', 'SAN JUAN NEP.']


In [19]:
lista_props = ['EX CHAVEZ']

In [32]:
cods_error

[]

In [33]:
len(lista_props)

7

In [34]:
df_intersects = gpd.read_file('INTERSECT_0.shp')

In [35]:
df_intersects.shape

(0, 16)

In [36]:
cods_error = []

In [37]:
contador = 0

for cod_prop in lista_props:
    print('inicio:', cod_prop)
    propiedad = ee.FeatureCollection(ruta_catastro)\
                .filter(ee.Filter.eq(CODIGO_PROPIEDAD, cod_prop))
    lotes_canha = propiedad
    NDVI = ee.Image(ruta_ndvi)
    NDVI_clip = NDVI.clip(lotes_canha.geometry())
    # crea un sample de los valore de pixel del NDVI
    NDVI_values = NDVI_clip.sampleRegions(lotes_canha.geometry())
    # reduce el resultado a valores de NDVI
    pixel_values = NDVI_values.reduceColumns(ee.Reducer.toList(),['b1']).get('list').getInfo()
    # aplica metodo Jenks
    breaks = jenkspy.jenks_breaks(pixel_values, n_classes=8)
    NDVI_class = ee.Image(-1).where(NDVI.lt(breaks[1]),1)\
                        .where(NDVI.gte(breaks[1]),2)\
                        .where(NDVI.gte(breaks[2]),3)\
                        .where(NDVI.gte(breaks[3]),4)\
                        .where(NDVI.gte(breaks[4]),5)\
                        .where(NDVI.gte(breaks[5]),6)\
                        .where(NDVI.gte(breaks[6]),7)\
                        .where(NDVI.gte(breaks[7]),8)
    NDVI_class_clip = NDVI_class.clip(lotes_canha.geometry())
    clasify = NDVI_class_clip.reproject(crs="EPSG:32720", scale=10)
    vector = clasify.reduceToVectors(**{
        'geometry': lotes_canha.geometry(),
        'crs': clasify.projection(),
        'scale': 10,
        'geometryType': 'polygon',
        'eightConnected': False
    })
    
    DF_VECTOR = vector
    
    lotes_local = geemap.ee_to_gdf(lotes_canha)
    
    lista = vector.toList(vector.size()).getInfo()
    lista_vector=[]
    for item in lista:
        dic = {'geometry':Polygon(item['geometry']['coordinates'][0]), 'count':item['properties']['count'], 'label':item['properties']['label']}
        lista_vector.append(dic)
    
    vector_local = gpd.GeoDataFrame(lista_vector)
    
    try:
        intersect = overlay(lotes_local, vector_local, how="intersection")
    except TypeError as e:
        cods_error.append(cod_prop)
        continue

    intersect.crs = "EPSG:4326"
    intersect = intersect.to_crs(epsg=32720)
    intersect['area_2'] = intersect['geometry'].area/10000
    
    area_01 = intersect['area_2'].sum()
    area_02 = lotes_local[SUPERFICIE].sum()
    
    area_diff = (area_02 - area_01)/len(intersect)
    intersect['area_2'] = intersect['area_2'] + area_diff
    
    DF = intersect.copy()
    dina = pd.pivot_table(DF, values='area_2', index=[CODIGO_PROPIEDAD, 'label'], aggfunc=np.sum)
    dina2 = dina.reset_index()
    
    dina2['tch'] = dina2['label'].apply(asig_cat)
    
    # Cargar el archivo de Excel existente
    wb = openpyxl.load_workbook('DATA_EST.xlsx')
    # Seleccionar la hoja de trabajo
    ws = wb['data']
    
    for i in range(0, len(dina2)):
        new_row = [dina2.iloc[i,0], dina2.iloc[i,1], dina2.iloc[i,2], dina2.iloc[i,3]]
        ws.append(new_row)
    wb.save('DATA_EST.xlsx')
    
    df_intersects = pd.concat([df_intersects, DF], ignore_index=True)
    #DF.to_file("INTERSECT_0.shp", driver="ESRI Shapefile")
    
    contador = contador + 1 
    print('fin: ', cod_prop)
    print('___________________CONTADOR: ', contador)
    
print('FIN......!')

inicio: BENITEZ 2


C:\Users\bismarksr\AppData\Local\Temp\ipykernel_4660\3655867732.py:63: FutureWarning: The provided callable <function sum at 0x0000026A8A7AEE60> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  dina = pd.pivot_table(DF, values='area_2', index=[CODIGO_PROPIEDAD, 'label'], aggfunc=np.sum)
C:\Users\bismarksr\AppData\Local\Temp\ipykernel_4660\3655867732.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_intersects = pd.concat([df_intersects, DF], ignore_index=True)


fin:  BENITEZ 2
___________________CONTADOR:  1
inicio: EX CHAVEZ


C:\Users\bismarksr\AppData\Local\Temp\ipykernel_4660\3655867732.py:63: FutureWarning: The provided callable <function sum at 0x0000026A8A7AEE60> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  dina = pd.pivot_table(DF, values='area_2', index=[CODIGO_PROPIEDAD, 'label'], aggfunc=np.sum)


fin:  EX CHAVEZ
___________________CONTADOR:  2
inicio: SARUBI


...

fin:  JARANA
___________________CONTADOR:  5
inicio: BUENA VISTA


C:\Users\bismarksr\AppData\Local\Temp\ipykernel_4660\3655867732.py:47: UserWarning: `keep_geom_type=True` in overlay resulted in 3 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  intersect = overlay(lotes_local, vector_local, how="intersection")
C:\Users\bismarksr\AppData\Local\Temp\ipykernel_4660\3655867732.py:63: FutureWarning: The provided callable <function sum at 0x0000026A8A7AEE60> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  dina = pd.pivot_table(DF, values='area_2', index=[CODIGO_PROPIEDAD, 'label'], aggfunc=np.sum)


fin:  BUENA VISTA
___________________CONTADOR:  6
inicio: SAN JUAN NEP.


C:\Users\bismarksr\AppData\Local\Temp\ipykernel_4660\3655867732.py:63: FutureWarning: The provided callable <function sum at 0x0000026A8A7AEE60> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  dina = pd.pivot_table(DF, values='area_2', index=[CODIGO_PROPIEDAD, 'label'], aggfunc=np.sum)


fin:  SAN JUAN NEP.
___________________CONTADOR:  7
FIN......!


In [38]:
cods_error

[]

In [39]:
df_intersects.tail(5)

,Corte,Fecha cort,MES,area,chacra,codparcela,ha_parcela,idparcela,super,tipo_culti,variedad,zafra,count,label,area_2,geometry,BLOQUES,Canal,PARCELA,idparecla
66227,1.0,NaN,,NaN,SAN JUAN NEP.,0,5.09,NaN,NaN,Caña,CTC-4,2025,6,3,0.056144,"POLYGON ((1203240.000 7090421.107, 1203240.000...",B,,24,B24
66228,1.0,NaN,,NaN,SAN JUAN NEP.,0,5.09,NaN,NaN,Caña,CTC-4,2025,14,4,0.132924,"POLYGON ((1203220.000 7090467.948, 1203220.000...",B,,24,B24
66229,1.0,NaN,,NaN,SAN JUAN NEP.,0,5.09,NaN,NaN,Caña,CTC-4,2025,62,3,0.648778,"POLYGON ((1203210.000 7090580.000, 1203210.000...",B,,24,B24
66230,1.0,NaN,,NaN,SAN JUAN NEP.,0,5.09,NaN,NaN,Caña,CTC-4,2025,4,2,0.037451,"POLYGON ((1203270.000 7090543.091, 1203270.000...",B,,24,B24
66231,1.0,NaN,,NaN,SAN JUAN NEP.,0,5.09,NaN,NaN,Caña,CTC-4,2025,5,1,0.026117,"POLYGON ((1203275.778 7090349.182, 1203275.119...",B,,24,B24


In [41]:
len(df_intersects)

66232

In [44]:
df_intersects.to_file("INTERSECT_01.shp", driver="ESRI Shapefile")

In [42]:
df_intersects['area_2'].sum()

4598.363

In [43]:
df_intersects.head(3)

,Corte,Fecha cort,MES,area,chacra,codparcela,ha_parcela,idparcela,super,tipo_culti,variedad,zafra,count,label,area_2,geometry,BLOQUES,Canal,PARCELA,idparecla
0,0,NaN,,NaN,BENITEZ 2,0,6.679,NaN,NaN,Caña,CTC-4,2025,3,6,0.018729,"POLYGON ((1178352.908 7093390.000, 1178360.000...",A,,12,A12
1,0,NaN,,NaN,BENITEZ 2,0,6.679,NaN,NaN,Caña,CTC-4,2025,14,7,0.134155,"POLYGON ((1178350.000 7093433.417, 1178350.000...",A,,12,A12
2,0,NaN,,NaN,BENITEZ 2,0,6.679,NaN,NaN,Caña,CTC-4,2025,1,8,0.010469,"POLYGON ((1178360.000 7093450.000, 1178360.000...",A,,12,A12
